In [ ]:
import numpy as np
import pandas as pd

In [ ]:
!gdown 1WI-TkvTXywmewllroaxXwuBh5aCFMtne

In [ ]:
train_df = pd.read_csv("data.csv")
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df['Gender']= train_df['Gender'].map({'Male':0, 'Female':1})
train_df['Married']= train_df['Married'].map({'No':0, 'Yes':1})
train_df['Loan_Status']= train_df['Loan_Status'].map({'N':0, 'Y':1})

In [ ]:
train_df.info()

In [ ]:
train_df.isnull().sum()

In [ ]:
## dropping all the missing values
train_df = train_df.dropna()
train_df.isnull().sum()

In [ ]:
X = train_df[['Gender', 'Married', 'ApplicantIncome', 'LoanAmount', 'Credit_History']]
y = train_df.Loan_Status
X.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=5)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(max_depth=4, random_state=5)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

pred_val = model.predict(X_val)
accuracy_score(y_val, pred_val)

In [ ]:
pred_train = model.predict(X_train)
accuracy_score(y_train, pred_train)

## MLFlow

In [ ]:
import mlflow
import os

In [ ]:
mlflow.set_experiment("loan_status")

In [ ]:
with mlflow.start_run():
    max_depth = 7
    model_rf = RandomForestClassifier(max_depth = max_depth, random_state = 5)
    model_rf.fit(X_train, y_train)

    pred_val = model_rf.predict(X_val)
    val_acc = accuracy_score(y_val, pred_val)
    pred_train = model_rf.predict(X_train)
    train_acc = accuracy_score(y_train, pred_train)

    mlflow.set_tag('mlflow.runName', '5th_run')
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_metric('val_acc', val_acc)
    mlflow.log_metric('train_acc', train_acc)

    mlflow.sklearn.log_model(model_rf, "model")

In [ ]:
def track_experiment(i, max_depth, random_state):
    with mlflow.start_run():
        model_rf = RandomForestClassifier(max_depth = max_depth, random_state = random_state)
        model_rf.fit(X_train, y_train)
    
        pred_val = model_rf.predict(X_val)
        val_acc = accuracy_score(y_val, pred_val)
        pred_train = model_rf.predict(X_train)
        train_acc = accuracy_score(y_train, pred_train)

        run_name = f"Run_Number_{i}"
        mlflow.set_tag('mlflow.runName', run_name)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("random_state", random_state)
        mlflow.log_metric('val_acc', val_acc)
        mlflow.log_metric('train_acc', train_acc)
    
        mlflow.sklearn.log_model(model_rf, "model")

In [ ]:
import random

for d in range(1,10):
    r = random.randint(5, 90)
    track_experiment(d, d, r)